In [1]:
from pathlib import Path
import sys

repo_root = Path.cwd().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from core.HMModel_def import HMModel
from core.EmissionSet_and_HiddenState_defs import EmissionSet, HiddenState
import math

# project 8 notebook

A hidden Markov model is a probabilistic sequence model with two layers:
- a sequence of hidden states, which are not observed directly, but are assumed to generate through a Markov process. 
- a sequence of observations whose probabilities depend on the current hidden state in a known way.

In a first-order Markov process, the hidden state at position `i` depends only on the hidden state at position `i-1`.

We can construct a first-order HMM object using the classes `EmissionSet`, `HiddenState`, and `HMModel`, imported from `/core`.

- `EmissionSet` defines one observation category, including its allowed values and default weights.
- `HiddenState` defines one hidden state, including its initial weight and emission weights for each emission set.
- `HMModel` stores the model-wide emission schema, hidden states, and hidden-to-hidden transition weights, and can derive normalized probability objects when needed.

Note that in this implementation, observations can be split across one or more emission sets. Each emission set represents one category of mutually exclusive observed values. The total emission probability for one position is then treated as the product of the probabilities from each emission set, conditional only on the current hidden state.

This object design is meant to be extensible: emission definitions, hidden-state definitions, and model structure can be edited independently while keeping the overall schema consistent.

## Viterbi algorithm

The Viterbi algorithm finds the single most likely hidden state path for a given observed sequence under a HMM. Rather than summing over all possible hidden state paths, it uses dynamic programming to keep only the best-scoring partial path ending in each state at each position. The traceback step then reconstructs which previous state produced the best score at each position.

The function below is an example implementation of the Viterbi algorithm using the current `HMModel` class.

Its main steps are:
- normalize the model so that probabilities are available (since our `HMModel` object primarily stores weights)
- validate and standardize the input 
- convert observed value names to integer indices for faster repeated lookup
- precompute log-transformed initial, transition, and emission probabilities
- iterate from left to right across the sequence, storing only:
  - the best previous score for each state
  - a traceback table indicating which previous state achieved that score
- reconstruct the optimal hidden-state path at the end

Log probabilities are used to avoid numerical underflow from repeated multiplication of small probabilities.

Note:
- For multiple emission sets, each observation is expected to be a list of strings (observed values).
- This workflow assumes a first-order HMM: transition probabilities can only be $P(s_i \mid s_{i-1})$.

In [2]:
def viterbi(emissions: list, model: HMModel) -> list[str]:
    
    #### SETUP AND VALIDATION ####
    model.normalize_all() # make sure probabilities exist
    states = [hs.hidden_state_name for hs in model.hidden_states]
    emission_sets = model.emission_sets
    n_states = len(states)
    n_sets = len(emission_sets)
    n = len(emissions)

    # standardize emissions shape
    if n_sets == 1: 
        # accept list, convert list to list of lists
        emissions = [[obs] for obs in emissions]
    else:
        # require each observation to be a list of strings of length n_sets
        for obs in emissions: 
            if not isinstance(obs, list):
                raise Exception("for multiple emission sets, each observation must be a list matching model.emission_sets order")
            if len(obs) != n_sets:
                raise Exception(f"each observation must have length {n_sets} ")

    # safe log probability helper
    def logp(p: float) -> float:
        if p <= 0:
            return float("-inf")
        return math.log(p)

    #### PRECOMPUTE LOOKUP OBJECTS ####
    
    # all observations can be represented as a list of indices 
    # ie. if coded_emissions[0] = [1,2], then the first observation had value 1 for the first es, value 2 for the second
    value_to_index = [{value_name: j for j, value_name in enumerate(es.value_names)} for es in emission_sets]
    # (it's better to make the map for each emission set than to scan with .index() in the next loop)
    coded_emissions = []
    for obs in emissions:
        coded_obs = []
        for i, value in enumerate(obs):
            coded_obs.append(value_to_index[i][value])
        coded_emissions.append(coded_obs)

    # init_logprobs[hs index] = init log prob for that hs
    init_logprobs = [logp(model.P_init[state_name]) for state_name in states]
    
    # trans_logprobs[previous state index][current state index] = trans log prob for those hs
    trans_logprobs = [
        [logp(model.P_hh[prev_state][curr_state]) for curr_state in states]
        for prev_state in states
        ]

    # emit_logtables[es index][hs index][value index] = log prob of that emission-set value given that hs
    emit_logtables = []
    for x, es in enumerate(emission_sets):
        set_name = es.set_name
        emit_logtables.append(
            [
                [logp(p) for p in model.P_eh[set_name][state_name]]
                for state_name in states
            ]
        )

    # empty traceback lookup
    traceback = [[None] * n for _ in range(n_states)]

    # we are only ever concerned with the previous "column" of scores
    # empty container for that:
    prev_scores = [float("-inf")] * n_states

    #### VITERBI ####
    # conceptually: at each position, we rebuild prev_scores and move forward one position

    # P(s_0) = P_init(s) x P(emission|s)
    # we compute the probabilities for each s_0
    first_obs = coded_emissions[0]
    for s in range(n_states):
        log_emit = 0.0                                                      # log(1) = 0

        # assuming conditional independence, 
        # we can add the log probs of obs from each set for a total "emission log prob"
        for x in range(n_sets):
            obs_index = first_obs[x]                                        # get observation for each emission set
            log_emit += emit_logtables[x][s][obs_index]
        
        prev_scores[s] = init_logprobs[s] + log_emit

    # when i > 0:
    # best score ending in s_i = best previous path score + transition log prob + emission log prob
    # for each s_i, we vary s_i-1 and keep the maximal score
    # we also keep track of what s_i-1 provided that maximum score (traceback)

    for i in range(1, n):                                                   # for position i...
        obs = coded_emissions[i]                                            # these are the observations

        # precompute P(emission|s) for this position
        curr_emit = [float("-inf")] * n_states
        for s in range(n_states):
            log_emit = 0.0                                                  # log(1) = 0
            for x in range(n_sets):
                obs_index = obs[x]                                          # get observation for each emission set
                log_emit += emit_logtables[x][s][obs_index]
            curr_emit[s] = log_emit                                         

        curr_scores = [float("-inf")] * n_states
        for curr_s in range(n_states):
            best_score = float("-inf")
            best_prev = None

            eprob = curr_emit[curr_s]

            for prev_s in range(n_states):
                score = prev_scores[prev_s] + trans_logprobs[prev_s][curr_s] + eprob
                # P(s_i|s_i-1) = P(s-1) x P_transition(s_i-1,s_i) x P(emission|s_i)

                if score > best_score:
                    best_score = score
                    best_prev = prev_s

            curr_scores[curr_s] = best_score
            traceback[curr_s][i] = best_prev

        prev_scores = curr_scores

    # termination
    best_last = max(range(n_states), key=lambda s: prev_scores[s])  # what's the "row number" of the best prev_score?

    #### TRACEBACK ####
    path = [best_last]

    for i in range(n - 1, 0, -1):
        path.append(traceback[path[-1]][i]) # what previous state allowed the best score for this state?
    
    path.reverse()
    
    return [states[s] for s in path]

## Demonstration

We build some conveniently shaped fake data:

In [3]:
# emission sets
dna_base = EmissionSet(
    name="dna_base",
    length=4,
    value_names=["A", "T", "C", "G"],
    default_weights=[1, 1, 1, 1]
    )

methylation = EmissionSet(
    name="methylation",
    length=2,
    value_names=["unmethylated", "methylated"],
    default_weights=[2, 1]
    )

# hidden states
background = HiddenState(
    name="background",
    init_weight=3.0,
    emission_weights={
        "dna_base": [2, 2, 1, 1],
        "methylation": [9, 1]
    }
)

CpG_island = HiddenState(
    name="CpG_island",
    init_weight=1.0,
    emission_weights={
        "dna_base": [1, 1, 5, 5],
        "methylation": [1, 9]
    }
)

# build model
model = HMModel(
    emission_sets=[dna_base, methylation],
    hidden_states=[background, CpG_island]
    )

# specify transition weights
model.replace_transition_row("background", [9, 1])
model.replace_transition_row("CpG_island", [1, 9])

# fake observations

emissions = [
    ["A", "unmethylated"],
    ["T", "unmethylated"],
    ["C", "methylated"],
    ["G", "methylated"],
    ["C", "methylated"],
    ["A", "unmethylated"],
    ["T", "unmethylated"],
    ["C", "unmethylated"],
    ["C", "unmethylated"],
    ["C", "methylated"],
    ["T", "methylated"],
    ["C", "methylated"],
    ["G", "methylated"],
    ["C", "methylated"],
]

Applying the `viterbi()` function returns the most likely hidden state path for the observed sequence:

In [4]:
viterbi(emissions, model)

['background',
 'background',
 'CpG_island',
 'CpG_island',
 'CpG_island',
 'background',
 'background',
 'background',
 'background',
 'CpG_island',
 'CpG_island',
 'CpG_island',
 'CpG_island',
 'CpG_island']

In this example, the middle region of the sequence contains several C and G bases paired with `methylated`, so the Viterbi path should tend to switch into `CpG_island` for that region. Positions with `A` or `T` and `unmethylated` should favor `background`. However, the decoded path is not determined by each position independently: the transition matrix strongly favors self-transitions. The algorithm balances how well the current observation fits each hidden state with how costly it is to switch from the previous hidden state. 

The current object classes and Viterbi implementation are hardcoded for a first-order hidden Markov model ie. $P(s_i \mid s_1,\dots,s_{i-1}) = P(s_i \mid s_{i-1})$. Extending this framework to higher-order Markov structure would require changing both the model representation and the algorithm. An nth-order HMM would need transition probabilities of the form $P(s_i \mid s_{i-1}, s_{i-2},...,s_{i-n})$ and Viterbi would need to track ordered combinations of states rather than single previous states.